In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.applications import ResNet50
import matplotlib.pyplot as plt
import requests
import tarfile
import os

In [2]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5vTzHBmQUaRNJQe5szCyKw/images-dataSAT.tar"
target_path = 'satellite_data.tar'


# Download the dataset
print("Downloading dataset...")
response = requests.get(url, stream=True)
if response.status_code == 200:
    with open(target_path, 'wb') as f:
        f.write(response.raw.read())


# Extract the tar file
print("Extracting files...")
with tarfile.open(target_path) as tar:
    tar.extractall(path="./data")
   
image_folder = "./data/images_dataSAT" # Adjust this based on extracted folder name
print("Data ready.")


Extracting files...
Data ready.


In [3]:
batch_size = 32 # How many images to look at once
image_size = (224, 224) # Resize all images to be the same size

print("Loading training data...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
image_folder,
validation_split=0.2, # Save 20% of the data for testing
subset="training", # This is the 80% used for training
seed=123, # A random seed ensures we get the same split every time
image_size=image_size,
batch_size=batch_size)

print("Loading testing data...")
test_dataset = tf.keras.utils.image_dataset_from_directory(
image_folder,
validation_split=0.2,
subset="validation", # This is the 20% used for testing
seed=123,
image_size=image_size,
batch_size=batch_size)

Loading training data...
Found 6000 files belonging to 2 classes.
Using 4800 files for training.
Loading testing data...
Found 6000 files belonging to 2 classes.
Using 1200 files for validation.


In [4]:
# Find out what our categories (classes) are
class_names = train_dataset.class_names
print("Categories found:", class_names)

Categories found: ['class_0_non_agri', 'class_1_agri']


In [7]:
# We will use "Transfer Learning". This means we take a model that has already 
# studied millions of images (ResNet50) and just teach it our specific satellite images.

# Load the pre-trained ResNet50, but leave off its final guessing layer (include_top=False)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model so we don't accidentally ruin what it already knows
base_model.trainable = False

# Build our new model by stacking layers like lego blocks
model = Sequential([
    base_model,                     # The pre-trained brain
    Flatten(),                      # Flatten the 3D image data into a 1D list
    Dense(128, activation='relu'),  # A layer of 128 "neurons" to learn patterns
    Dropout(0.2),                   # Randomly turn off 20% of neurons to prevent memorizing
    Dense(len(class_names), activation='softmax') # The final layer that makes the guess
])

# Tell the model how to learn
model.compile(
    optimizer='adam', # The standard tool used to update the model's guesses
    loss='sparse_categorical_crossentropy', # The math used to calculate how wrong it is
    metrics=['accuracy'] # We want to track the accuracy percentage
)

In [8]:
print("Starting training...")
# "Fit" is the command that actually makes the model study the images
epochs = 5 # How many times to read through the entire dataset
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=epochs
)

Starting training...
Epoch 1/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 193s 1s/step - accuracy: 0.9421 - loss: 1.1058 - val_accuracy: 0.9867 - val_loss: 0.0528
Epoch 2/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 185s 1s/step - accuracy: 0.9887 - loss: 0.0403 - val_accuracy: 0.9833 - val_loss: 0.0474
Epoch 3/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 187s 1s/step - accuracy: 0.9937 - loss: 0.0287 - val_accuracy: 0.9825 - val_loss: 0.0757
Epoch 4/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 184s 1s/step - accuracy: 0.9952 - loss: 0.0247 - val_accuracy: 0.9825 - val_loss: 0.0672
Epoch 5/5
150/150 ━━━━━━━━━━━━━━━━━━━━ 185s 1s/step - accuracy: 0.9940 - loss: 0.0238 - val_accuracy: 0.9833 - val_loss: 0.0593


In [9]:
# Let's print out the final score on the test data
loss, accuracy = model.evaluate(test_dataset)
print(f"Final Test Accuracy: {accuracy * 100:.2f}%")

38/38 ━━━━━━━━━━━━━━━━━━━━ 36s 935ms/step - accuracy: 0.9833 - loss: 0.0593
Final Test Accuracy: 98.33%


In [11]:
from tensorflow.keras.preprocessing import image
import numpy as np
# Load a new image that the model has NEVER seen
# Replace 'path_to_new_image.jpg' with an actual image file on your computer
img_path = 'Non Agri Image.jpg' 
img = image.load_img(img_path, target_size=(224, 224))

# 2. Convert the image to numbers so the model can read it
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) # Create a "batch" of one image

# 3. Make the prediction!
predictions = model.predict(img_array)

# 4. Find the category with the highest score
score = tf.nn.softmax(predictions[0])
predicted_class = class_names[np.argmax(score)]
confidence = 100 * np.max(score)
print(f"This image is most likely '{predicted_class}' with a {confidence:.2f}% confidence.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
This image is most likely 'class_1_agri' with a 73.11% confidence.
